In [ ]:
# ============================================================
# EXEKUCE NA ORP
# počet osob v exekuci / počet obyvatel ORP
# ROBUSTNÍ VERZE: populace se bere z obecní tabulky ČSÚ
# a agreguje se do ORP přes ZÚJ -> ORP
# ============================================================

# Pokud něco chybí, odkomentuj:
# %pip install pandas geopandas matplotlib requests pyogrio fiona shapely openpyxl

from pathlib import Path
import zipfile
import warnings

import requests
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# ============================================================
# 1) NASTAVENÍ
# ============================================================

DATA_DIR = Path("data_exekuce_orp")
DATA_DIR.mkdir(exist_ok=True)

# EKČR – otevřená data
EXEKUCE_URL = "https://statistiky.ekcr.info/otevrena-data/data/povinni.json"

# ČSÚ – obecní tabulka (Tab. 3) z produktu "Počet obyvatel v obcích - k 1. 1. 2025"
# Schválně používáme obecní tabulku, ne ORP tabulku.
POP_OBEC_XLSX_URL = "https://csu.gov.cz/docs/107508/ebed5ef3-dca1-baf2-c0d3-824b0893086f/1300722503.xlsx?version=1.0"

# RÚIAN / ČÚZK – nechávám stejný ZIP, který už se ti stahoval
RUIAN_ZIP_URL = "https://services.cuzk.gov.cz/shp/stat/epsg-5514/1.zip"

EXEKUCE_FILE = DATA_DIR / "povinni.json"
POP_OBEC_FILE = DATA_DIR / "csu_populace_obce_2025.xlsx"
RUIAN_ZIP_FILE = DATA_DIR / "ruian_stat_1.zip"
RUIAN_DIR = DATA_DIR / "ruian_unzipped"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) Python requests"
}

# ============================================================
# 2) POMOCNÉ FUNKCE
# ============================================================

def download_file(url: str, dest: Path, force: bool = False) -> None:
    if dest.exists() and not force:
        print(f"Soubor už existuje: {dest.name}")
        return

    print(f"Stahuji: {url}")
    with requests.get(url, headers=HEADERS, stream=True, timeout=(30, 300)) as r:
        r.raise_for_status()
        with open(dest, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
    print(f"Uloženo: {dest}")


def unzip_file(zip_path: Path, out_dir: Path) -> None:
    out_dir.mkdir(exist_ok=True)

    shp_files = list(out_dir.rglob("*.shp"))
    if shp_files:
        print(f"ZIP už je rozbalený v: {out_dir}")
        return

    print(f"Rozbaluji ZIP: {zip_path.name}")
    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)
    print(f"Rozbaleno do: {out_dir}")


def normalize_numeric(series: pd.Series) -> pd.Series:
    return pd.to_numeric(
        series.astype(str)
              .str.replace("\xa0", "", regex=False)
              .str.replace(" ", "", regex=False)
              .str.replace(",", ".", regex=False),
        errors="coerce"
    )


def find_shp_file(base_dir: Path, preferred_names):
    shp_files = list(base_dir.rglob("*.shp"))
    if not shp_files:
        raise FileNotFoundError("V rozbalené složce nebyl nalezen žádný .shp soubor.")

    shp_map = {p.name.upper(): p for p in shp_files}

    for name in preferred_names:
        if name.upper() in shp_map:
            return shp_map[name.upper()]

    for p in shp_files:
        pu = p.name.upper()
        for name in preferred_names:
            token = name.upper().replace(".SHP", "")
            if token in pu:
                return p

    raise FileNotFoundError(
        f"Nepodařilo se najít shapefile. Hledal jsem: {preferred_names}\n"
        f"Nalezené soubory: {[p.name for p in shp_files]}"
    )


def read_vector(path: Path) -> gpd.GeoDataFrame:
    attempts = [
        {"engine": "pyogrio"},
        {"engine": "pyogrio", "encoding": "cp1250"},
        {},
        {"encoding": "cp1250"},
    ]

    last_error = None
    for kwargs in attempts:
        try:
            return gpd.read_file(path, **kwargs)
        except Exception as e:
            last_error = e

    raise RuntimeError(f"Nepodařilo se načíst {path}. Poslední chyba: {last_error}")


def pick_col(columns, candidates):
    cols = list(columns)
    cols_upper = {str(c).upper(): c for c in cols}

    for cand in candidates:
        if cand.upper() in cols_upper:
            return cols_upper[cand.upper()]

    for cand in candidates:
        for c in cols:
            cu = str(c).upper()
            if cand.upper() in cu or cu.startswith(cand.upper()):
                return c

    raise KeyError(f"Nepodařilo se najít sloupec {candidates}. Dostupné sloupce: {cols}")


def load_population_obec_from_xlsx(xlsx_path: Path) -> pd.DataFrame:
    """
    Robustně načte obecní tabulku ČSÚ.
    Nehledá konkrétní skiprows ani konkrétní názvy sloupců:
    zkouší kombinace sloupců a vybere tu, která vypadá jako tabulka obcí.

    Očekává:
    - 1 sloupec = ZÚJ obce (typicky 6místný kód)
    - 2 sloupec = název obce
    - některý další sloupec = počet obyvatel celkem
    """
    xls = pd.ExcelFile(xlsx_path, engine="openpyxl")

    best_df = None
    best_meta = None

    for sheet in xls.sheet_names:
        raw = pd.read_excel(xls, sheet_name=sheet, header=None)
        ncols = raw.shape[1]

        # prohledáme první sloupce; obecní tabulka bývá vlevo
        max_code_col = min(8, max(1, ncols - 2))

        for code_idx in range(max_code_col):
            if code_idx + 2 >= ncols:
                continue

            code = normalize_numeric(raw.iloc[:, code_idx])
            name = raw.iloc[:, code_idx + 1].astype(str).str.strip()

            # ZÚJ obcí jsou typicky 6místné kódy
            base_mask = (
                code.notna() &
                code.between(100000, 999999) &
                name.notna() &
                (name.str.lower() != "nan")
            )

            # zkusíme několik následujících numerických sloupců jako kandidáta na "celkem obyvatel"
            for pop_idx in range(code_idx + 2, min(code_idx + 8, ncols)):
                pop = normalize_numeric(raw.iloc[:, pop_idx])

                cand = pd.DataFrame({
                    "kod_obce_zuj": code,
                    "nazev_obce_csu": name,
                    "obyvatele_obec": pop
                })

                cand = cand[
                    base_mask &
                    cand["obyvatele_obec"].notna() &
                    (cand["obyvatele_obec"] > 0)
                ].copy()

                # odfiltruj řádky, které vypadají jako hlavičky/patičky
                bad_pattern = (
                    r"^zuj$|^kod$|^obec$|^název$|^nazev$|"
                    r"průměr|prum|věk|vek|muži|muzi|ženy|zeny|"
                    r"okres|kraj|republika|celkem|lau|nuts"
                )
                cand = cand[
                    ~cand["nazev_obce_csu"].str.lower().str.contains(bad_pattern, regex=True, na=False)
                ].copy()

                # deduplikace podle ZÚJ
                cand["kod_obce_zuj"] = cand["kod_obce_zuj"].astype("Int64")
                cand = cand.dropna(subset=["kod_obce_zuj"])
                cand = cand.drop_duplicates(subset=["kod_obce_zuj"], keep="first")

                row_count = len(cand)
                pop_sum = cand["obyvatele_obec"].sum()

                # správná tabulka by měla mít tisíce obcí
                if row_count < 5000:
                    continue

                meta = {
                    "sheet": sheet,
                    "code_idx": code_idx,
                    "pop_idx": pop_idx,
                    "rows": row_count,
                    "pop_sum": pop_sum
                }

                if best_meta is None:
                    best_df = cand.copy()
                    best_meta = meta
                else:
                    # preferuj víc řádků; při shodě vyšší součet populace
                    if (
                        meta["rows"] > best_meta["rows"] or
                        (meta["rows"] == best_meta["rows"] and meta["pop_sum"] > best_meta["pop_sum"])
                    ):
                        best_df = cand.copy()
                        best_meta = meta

    if best_df is None:
        raise ValueError("Nepodařilo se najít obecní tabulku populace v XLSX z ČSÚ.")

    best_df["kod_obce_zuj"] = best_df["kod_obce_zuj"].astype(int)
    best_df["obyvatele_obec"] = normalize_numeric(best_df["obyvatele_obec"]).astype(int)
    best_df["nazev_obce_csu"] = best_df["nazev_obce_csu"].astype(str).str.strip()

    print("\nNalezena obecní tabulka ČSÚ:")
    print(best_meta)

    return best_df[["kod_obce_zuj", "nazev_obce_csu", "obyvatele_obec"]].reset_index(drop=True)


# ============================================================
# 3) STAŽENÍ DAT
# ============================================================

download_file(EXEKUCE_URL, EXEKUCE_FILE)
download_file(POP_OBEC_XLSX_URL, POP_OBEC_FILE)
download_file(RUIAN_ZIP_URL, RUIAN_ZIP_FILE)

# ============================================================
# 4) EXEKUCE -> počet osob v exekuci po obcích
# ============================================================

print("\nNačítám exekuce...")
exekuce = pd.read_json(EXEKUCE_FILE)

print("Sloupce v exekučních datech:")
print(exekuce.columns.tolist())

required_ex_cols = {"id", "kod_obce_zuj"}
missing_ex_cols = required_ex_cols - set(exekuce.columns)
if missing_ex_cols:
    raise KeyError(f"V exekučních datech chybí sloupce: {missing_ex_cols}")

exekuce["id"] = normalize_numeric(exekuce["id"])
exekuce["kod_obce_zuj"] = normalize_numeric(exekuce["kod_obce_zuj"])

# unikátní osoby v exekuci po obcích
exekuce_obec = (
    exekuce[["id", "kod_obce_zuj"]]
    .dropna()
    .drop_duplicates()
    .groupby("kod_obce_zuj", as_index=False)["id"]
    .nunique()
    .rename(columns={"id": "osoby_v_exekuci"})
)

exekuce_obec["kod_obce_zuj"] = exekuce_obec["kod_obce_zuj"].astype(int)

print(f"Počet obcí s exekučními daty: {len(exekuce_obec):,}")

# ============================================================
# 5) RÚIAN -> mapování OBEC -> ORP + geometrie ORP
# ============================================================

print("\nPracuji s RÚIAN...")
unzip_file(RUIAN_ZIP_FILE, RUIAN_DIR)

orp_shp = find_shp_file(RUIAN_DIR, ["ORP_P.shp", "ORP.shp"])
obec_shp = find_shp_file(RUIAN_DIR, ["OBEC_P.shp", "OBEC.shp", "OBCE_P.shp"])

print(f"Použitý ORP shapefile:  {orp_shp}")
print(f"Použitý OBEC shapefile: {obec_shp}")

orp_gdf = read_vector(orp_shp)
obec_gdf = read_vector(obec_shp)

print("\nSloupce ORP:")
print(orp_gdf.columns.tolist())

print("\nSloupce OBEC:")
print(obec_gdf.columns.tolist())

orp_kod_col = pick_col(orp_gdf.columns, ["KOD", "ORP_KOD"])
orp_name_col = pick_col(orp_gdf.columns, ["NAZEV"])

obec_kod_col = pick_col(obec_gdf.columns, ["KOD", "OBEC_KOD"])
obec_orp_col = pick_col(obec_gdf.columns, ["ORP_KOD"])

# vazba obec -> ORP
obec_to_orp = (
    obec_gdf[[obec_kod_col, obec_orp_col]]
    .copy()
    .rename(columns={
        obec_kod_col: "kod_obce_zuj",
        obec_orp_col: "orp_kod"
    })
    .drop_duplicates()
)

obec_to_orp["kod_obce_zuj"] = normalize_numeric(obec_to_orp["kod_obce_zuj"]).astype("Int64")
obec_to_orp["orp_kod"] = normalize_numeric(obec_to_orp["orp_kod"]).astype("Int64")
obec_to_orp = obec_to_orp.dropna(subset=["kod_obce_zuj", "orp_kod"]).copy()
obec_to_orp["kod_obce_zuj"] = obec_to_orp["kod_obce_zuj"].astype(int)
obec_to_orp["orp_kod"] = obec_to_orp["orp_kod"].astype(int)

# geometrie ORP
map_gdf = orp_gdf[[orp_kod_col, orp_name_col, orp_gdf.geometry.name]].copy()
map_gdf = map_gdf.rename(columns={
    orp_kod_col: "orp_kod",
    orp_name_col: "orp_nazev"
})
map_gdf["orp_kod"] = normalize_numeric(map_gdf["orp_kod"]).astype("Int64")
map_gdf = map_gdf.dropna(subset=["orp_kod"]).copy()
map_gdf["orp_kod"] = map_gdf["orp_kod"].astype(int)

# ============================================================
# 6) ČSÚ -> populace obcí
# ============================================================

print("\nNačítám populaci obcí z ČSÚ...")
pop_obec = load_population_obec_from_xlsx(POP_OBEC_FILE)

print(f"Počet obcí v populačních datech: {len(pop_obec):,}")
print(pop_obec.head())

# ============================================================
# 7) AGREGACE DO ORP
# ============================================================

# exekuce: obec -> ORP
exekuce_orp = exekuce_obec.merge(obec_to_orp, on="kod_obce_zuj", how="left")

missing_ex_mapping = exekuce_orp["orp_kod"].isna().sum()
if missing_ex_mapping > 0:
    print(f"Pozor: {missing_ex_mapping} obcí z exekučních dat se nepodařilo přiřadit k ORP.")

exekuce_orp = (
    exekuce_orp.dropna(subset=["orp_kod"])
    .groupby("orp_kod", as_index=False)["osoby_v_exekuci"]
    .sum()
)
exekuce_orp["orp_kod"] = exekuce_orp["orp_kod"].astype(int)

# populace: obec -> ORP
pop_orp = pop_obec.merge(obec_to_orp, on="kod_obce_zuj", how="left")

missing_pop_mapping = pop_orp["orp_kod"].isna().sum()
if missing_pop_mapping > 0:
    print(f"Pozor: {missing_pop_mapping} obcí z populačních dat se nepodařilo přiřadit k ORP.")

pop_orp = (
    pop_orp.dropna(subset=["orp_kod"])
    .groupby("orp_kod", as_index=False)["obyvatele_obec"]
    .sum()
    .rename(columns={"obyvatele_obec": "obyvatele_orp"})
)
pop_orp["orp_kod"] = pop_orp["orp_kod"].astype(int)
pop_orp["obyvatele_orp"] = pop_orp["obyvatele_orp"].astype(int)

# ============================================================
# 8) SPOJENÍ DAT + VÝPOČET PODÍLU
# ============================================================

result_gdf = (
    map_gdf
    .merge(exekuce_orp, on="orp_kod", how="left")
    .merge(pop_orp, on="orp_kod", how="left")
)

result_gdf["osoby_v_exekuci"] = result_gdf["osoby_v_exekuci"].fillna(0)
result_gdf["podil_exekuce"] = result_gdf["osoby_v_exekuci"] / result_gdf["obyvatele_orp"]
result_gdf["podil_exekuce_pct"] = result_gdf["podil_exekuce"] * 100

result_tbl = (
    result_gdf[["orp_kod", "orp_nazev", "osoby_v_exekuci", "obyvatele_orp", "podil_exekuce_pct"]]
    .copy()
    .sort_values("podil_exekuce_pct", ascending=False)
)

print("\nTOP 15 ORP podle podílu osob v exekuci:")
print(
    result_tbl.head(15).to_string(
        index=False,
        formatters={
            "osoby_v_exekuci": lambda x: f"{x:,.0f}".replace(",", " "),
            "obyvatele_orp": lambda x: f"{x:,.0f}".replace(",", " ") if pd.notna(x) else "NaN",
            "podil_exekuce_pct": lambda x: f"{x:.2f} %" if pd.notna(x) else "NaN"
        }
    )
)

print("\nKontrola Prahy:")
print(
    result_tbl[
        result_tbl["orp_nazev"].astype(str).str.contains("Praha", case=False, na=False)
    ].to_string(
        index=False,
        formatters={
            "osoby_v_exekuci": lambda x: f"{x:,.0f}".replace(",", " "),
            "obyvatele_orp": lambda x: f"{x:,.0f}".replace(",", " ") if pd.notna(x) else "NaN",
            "podil_exekuce_pct": lambda x: f"{x:.2f} %" if pd.notna(x) else "NaN"
        }
    )
)

print("\nPočet ORP bez populace:", int(result_gdf["obyvatele_orp"].isna().sum()))
print("Počet ORP bez exekucí:", int(result_gdf["osoby_v_exekuci"].isna().sum()))

# ============================================================
# 9) INTERAKTIVNÍ MAPA ČR (FOLIUM) – verze pro JupyterLab
# ============================================================

# pokud chybí:
# %pip install folium branca

import folium
import branca.colormap as cm
from branca.element import Figure
from IPython.display import display

# Folium chce WGS84
map_interactive = result_gdf.to_crs(epsg=4326).copy()
map_interactive = map_interactive[map_interactive.geometry.notna()].copy()

# formátované texty
map_interactive["osoby_v_exekuci_fmt"] = map_interactive["osoby_v_exekuci"].fillna(0).map(
    lambda x: f"{int(round(x)):,}".replace(",", " ")
)

map_interactive["obyvatele_orp_fmt"] = map_interactive["obyvatele_orp"].map(
    lambda x: f"{int(round(x)):,}".replace(",", " ") if pd.notna(x) else "chybí"
)

map_interactive["podil_exekuce_pct_fmt"] = map_interactive["podil_exekuce_pct"].map(
    lambda x: f"{x:.2f} %" if pd.notna(x) else "chybí"
)

valid_vals = map_interactive["podil_exekuce_pct"].dropna()
if len(valid_vals) == 0:
    raise ValueError("Sloupec 'podil_exekuce_pct' neobsahuje žádné platné hodnoty.")

vmin = float(valid_vals.min())
vmax = float(valid_vals.max())

colormap = cm.linear.YlOrRd_09.scale(vmin, vmax)
colormap.caption = "Podíl osob v exekuci (%)"

fig = Figure(width=1000, height=700)

# bounding box ČR podle tvých ORP polygonů
minx, miny, maxx, maxy = map_interactive.total_bounds

# malé odsazení, aby mapa nebyla úplně "natěsno"
pad_x = (maxx - minx) * 0.02
pad_y = (maxy - miny) * 0.02

bounds = [
    [miny - pad_y, minx - pad_x],   # jihozápad [lat, lon]
    [maxy + pad_y, maxx + pad_x]    # severovýchod
]

m = folium.Map(
    tiles="CartoDB positron",
    zoom_control=True,
    max_bounds=True
)

# nastaví zobrazení přesně na ČR
m.fit_bounds(bounds)

# omezí posun mapy mimo ČR
m.options["maxBounds"] = bounds

fig.add_child(m)

def style_function(feature):
    value = feature["properties"]["podil_exekuce_pct"]
    if value is None:
        return {
            "fillColor": "#d3d3d3",
            "color": "white",
            "weight": 0.7,
            "fillOpacity": 0.7
        }
    return {
        "fillColor": colormap(value),
        "color": "white",
        "weight": 0.7,
        "fillOpacity": 0.85
    }

def highlight_function(feature):
    return {
        "weight": 2,
        "color": "#333333",
        "fillOpacity": 0.95
    }

tooltip = folium.GeoJsonTooltip(
    fields=[
        "orp_nazev",
        "osoby_v_exekuci_fmt",
        "obyvatele_orp_fmt",
        "podil_exekuce_pct_fmt"
    ],
    aliases=[
        "ORP:",
        "Osoby v exekuci:",
        "Počet obyvatel:",
        "Podíl:"
    ],
    sticky=False,
    labels=True
)

popup = folium.GeoJsonPopup(
    fields=[
        "orp_nazev",
        "orp_kod",
        "osoby_v_exekuci_fmt",
        "obyvatele_orp_fmt",
        "podil_exekuce_pct_fmt"
    ],
    aliases=[
        "ORP:",
        "Kód ORP:",
        "Osoby v exekuci:",
        "Počet obyvatel:",
        "Podíl:"
    ],
    labels=True
)

folium.GeoJson(
    map_interactive,
    name="Podíl osob v exekuci",
    style_function=style_function,
    highlight_function=highlight_function,
    tooltip=tooltip,
    popup=popup
).add_to(m)

colormap.add_to(m)
folium.LayerControl().add_to(m)

# ZOBRAZENÍ V NOTEBOOKU
display(fig)

# ULOŽENÍ DO HTML
html_out = DATA_DIR / "interaktivni_mapa_orp_exekuce.html"
m.save(str(html_out))
print(f"Interaktivní mapa uložena do: {html_out}")

# ============================================================
# 10) ULOŽENÍ VÝSLEDKŮ
# ============================================================

csv_out = DATA_DIR / "orp_exekuce_podil_pop_agregace_z_obci_2025.csv"
result_tbl.to_csv(csv_out, index=False, encoding="utf-8-sig")

print(f"\nCSV uloženo do: {csv_out}")

try:
    gpkg_out = DATA_DIR / "orp_exekuce_podil_pop_agregace_z_obci_2025.gpkg"
    result_gdf.to_file(gpkg_out, layer="orp_exekuce", driver="GPKG")
    print(f"GPKG uloženo do: {gpkg_out}")
except Exception as e:
    print(f"GPKG se nepodařilo uložit: {e}")